# Genie360 — 04: Grant App Permissions

Permission bootstrap notebook:
- Grants the Databricks App's service principal CAN USE on SQL Warehouse
- Grants CAN READ on system.query.history
- Grants CAN EDIT on target Genie Spaces
- Validates all permissions

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from genie360.pipeline import load_genie360_config

config = load_genie360_config(project_root / "genie360" / "config.yml")
print("✅ Config loaded")

In [ ]:
import os
from databricks.sdk import WorkspaceClient

w = WorkspaceClient(
    host=os.environ.get("DATABRICKS_HOST"),
    token=os.environ.get("DATABRICKS_TOKEN"),
)
print(f"✅ Connected to workspace: {w.config.host}")

## 1. Configure Service Principal

Set the service principal ID that the Genie360 Databricks App will use.

In [ ]:
# Replace with your Databricks App's service principal application ID
SERVICE_PRINCIPAL_APP_ID = ""  # e.g., "12345678-abcd-efgh-ijkl-123456789012"

# Target Genie Space IDs to grant access to
TARGET_SPACE_IDS = []  # e.g., ["space_id_1", "space_id_2"]

# SQL Warehouse ID
WAREHOUSE_ID = config.get("warehouse_id", "")

print(f"Service Principal: {SERVICE_PRINCIPAL_APP_ID}")
print(f"Target Spaces: {TARGET_SPACE_IDS}")
print(f"Warehouse: {WAREHOUSE_ID}")

## 2. Grant SQL Warehouse Access

In [ ]:
if WAREHOUSE_ID and SERVICE_PRINCIPAL_APP_ID:
    try:
        from databricks.sdk.service.sql import (
            SetWorkspaceWarehouseConfigRequest,
            EndpointPermissionLevel,
            EndpointAccessControlRequest,
        )

        w.warehouses.set_permissions(
            warehouse_id=WAREHOUSE_ID,
            access_control_list=[
                EndpointAccessControlRequest(
                    service_principal_name=SERVICE_PRINCIPAL_APP_ID,
                    permission_level=EndpointPermissionLevel.CAN_USE,
                )
            ],
        )
        print(f"✅ Granted CAN_USE on warehouse {WAREHOUSE_ID}")
    except Exception as e:
        print(f"❌ Failed to grant warehouse access: {e}")
else:
    print("⚠️ Set WAREHOUSE_ID and SERVICE_PRINCIPAL_APP_ID first")

## 3. Grant System Tables Access

In [ ]:
try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

if SERVICE_PRINCIPAL_APP_ID:
    grants = [
        "GRANT SELECT ON system.query.history TO `{sp}`",
        "GRANT SELECT ON system.billing.usage TO `{sp}`",
        "GRANT SELECT ON system.compute.clusters TO `{sp}`",
    ]
    for grant in grants:
        try:
            spark.sql(grant.format(sp=SERVICE_PRINCIPAL_APP_ID))
            print(f"✅ {grant.format(sp=SERVICE_PRINCIPAL_APP_ID)}")
        except Exception as e:
            print(f"⚠️ Grant may require admin: {e}")
else:
    print("⚠️ Set SERVICE_PRINCIPAL_APP_ID first")

## 4. Grant Genie Space Edit Access

In [ ]:
import requests

if SERVICE_PRINCIPAL_APP_ID and TARGET_SPACE_IDS:
    host = os.environ.get("DATABRICKS_HOST", "").rstrip("/")
    token = os.environ.get("DATABRICKS_TOKEN", "")

    for space_id in TARGET_SPACE_IDS:
        try:
            url = f"{host}/api/2.0/permissions/genie/spaces/{space_id}"
            headers = {"Authorization": f"Bearer {token}"}
            payload = {
                "access_control_list": [
                    {
                        "service_principal_name": SERVICE_PRINCIPAL_APP_ID,
                        "permission_level": "CAN_EDIT",
                    }
                ]
            }
            resp = requests.patch(url, headers=headers, json=payload, timeout=30)
            if resp.status_code == 200:
                print(f"✅ Granted CAN_EDIT on Genie Space {space_id}")
            else:
                print(f"⚠️ Space {space_id}: {resp.status_code} — {resp.text}")
        except Exception as e:
            print(f"❌ Space {space_id}: {e}")
else:
    print("⚠️ Set SERVICE_PRINCIPAL_APP_ID and TARGET_SPACE_IDS first")

## 5. Validate Permissions

In [ ]:
print("\n🔍 Permission Validation:\n")

# Verify system tables
try:
    spark.sql("SELECT 1 FROM system.query.history LIMIT 1").collect()
    print("  ✅ system.query.history — accessible")
except Exception as e:
    print(f"  ❌ system.query.history — {e}")

# Verify Genie Space API
if TARGET_SPACE_IDS:
    from genie360.modules.genie_space_injection import get_genie_space
    for space_id in TARGET_SPACE_IDS:
        try:
            space = get_genie_space(space_id)
            print(f"  ✅ Genie Space {space_id} — accessible")
        except Exception as e:
            print(f"  ❌ Genie Space {space_id} — {e}")

print("\n🎯 Permission bootstrap complete!")